In [2]:
import os

from c00_utils import *
# exec(open('c00_utils.py').read())

pn.extension()  # 启用 Jupyter 支持
pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 100)

In [3]:
native_exchange_data_dir = os.path.join(base_dir, "Native_Exchange_Data")
pro = ts.pro_api(secret_dict['tushare']['api_key'])

def get_exchange_variety_map():
    # 交易所日历路径 NED_dimension_exchange_calendar_path
    NED_dimension_exchange_calendar_path = os.path.join(native_exchange_data_dir, 'dimension_exchange_calendar.csv')
    dimension_exchange_calendar = pd.read_csv(NED_dimension_exchange_calendar_path, dtype = {'date': str})

    # 获取 交易所-品种 表单
    exchange_list = [col for col in dimension_exchange_calendar.columns if col != 'date'] # 交易所列表
    map_list = []
    for exchange in exchange_list:

        exchange_variety_map = pd.DataFrame()
        for attempt in range(3):
            try:
                exchange_variety_map = pro.fut_basic(exchange = exchange, fut_type = "1", fields = ["fut_code"])
                break
            except Exception as e:
                print(f"第 {attempt + 1} 次获取 {exchange} 品种信息失败: {e}")

        exchange_variety_map = exchange_variety_map.drop_duplicates()
        exchange_variety_map["exchange"] = exchange
        map_list.append(exchange_variety_map)

    exchange_variety_map = pd.concat(map_list, ignore_index = True)
    exchange_variety_map = exchange_variety_map.groupby('exchange')['fut_code'].apply(list)
    exchange_variety_map = exchange_variety_map.to_dict()
    return exchange_variety_map

if __name__ == '__main__':
    exchange_variety_map = get_exchange_variety_map()
    print(exchange_variety_map)

{'CFFEX': ['T', 'TF', 'IF', 'IM', 'IC', 'IH', 'TS', 'TL'], 'CZCE': ['TA', 'ZC', 'TC', 'WT', 'CJ', 'PL', 'SR', 'CF', 'CY', 'FG', 'JR', 'LR', 'MA', 'OI', 'PF', 'PM', 'RI', 'RM', 'RS', 'SA', 'SF', 'SM', 'UR', 'WH', 'AP', 'WS', 'SH', 'ME', 'RO', 'PK', 'ER', 'PR', 'PX'], 'DCE': ['JD', 'CS', 'JM', 'C', 'M', 'V', 'EB', 'EG', 'P', 'Y', 'L', 'FB', 'A', 'B', 'BB', 'I', 'J', 'PP', 'RR', 'LH', 'PG', 'LG', 'L_F', 'PP_F', 'V_F', 'BZ'], 'GFEX': ['LC', 'SI', 'PS', 'PD', 'PT'], 'INE': ['NR', 'SC', 'LU', 'BC', 'SCTAS', 'EC'], 'SHFE': ['AO', 'NI', 'AL', 'HC', 'BU', 'RU', 'FU', 'CU', 'AG', 'ZN', 'AU', 'PB', 'SS', 'RB', 'SN', 'WR', 'SP', 'BR', 'OP', 'AD']}


In [4]:
# # 品种文件夹路径 NED_variety_path
# NED_variety_path = os.path.join(native_exchange_data_dir, variety)
#
# # 合约日历路径 NED_dimension_symbol_calendar_path
# NED_dimension_symbol_calendar_path = os.path.join(NED_variety_path, 'dimension_symbol_calendar.csv')
#
#
#
#
# # 时间序列文件夹路径 NED_variety_timeseries_path
# NED_variety_timeseries_path = os.path.join(NED_variety_path , "timeseries_data")
#
# # 合约状态表路径 NED_dimension_symbol_state_path
# NED_dimension_symbol_state_path = os.path.join(NED_variety_timeseries_path, 'dimension_symbol_state.csv')
#
# # 分钟行情时间序列文件夹路径 NED_futures_minutely_timeseries_path
# NED_futures_minutely_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_minutely")
#
# # 分钟行情时间序列文件路径
# NED_fact_futures_minutely_timeseries_path = os.path.join(NED_futures_minutely_timeseries_path, f"{symbol}.csv")

In [4]:
allfutures_1min_varity_path_dict = dict()
allfutures_1min_path = r"E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟"

for exchange in os.listdir(allfutures_1min_path):
    exchange_path = os.path.join(allfutures_1min_path, exchange)
    print(exchange_path)

    for variety in os.listdir(exchange_path):
        variety_path = os.path.join(exchange_path, variety)
        print(variety_path)

        for symbol in os.listdir(variety_path):
            symbol_path = os.path.join(variety_path, symbol)

            # 检查是否是文件且以.csv结尾
            if os.path.isfile(symbol_path) and symbol.endswith('.csv'):
                # 分离文件名和扩展名
                filename, ext = os.path.splitext(symbol)

                # 如果文件名中包含点，去掉最后一个点及后面的内容
                if '.' in filename:
                    new_filename = filename.split('.')[0] + ext
                    new_symbol_path = os.path.join(variety_path, new_filename)

                    os.rename(symbol_path, new_symbol_path)
                    print(f"重命名: {symbol} -> {new_filename}") # 重命名

                    symbol_path = new_symbol_path

                allfutures_1min_varity_path_dict[variety] = allfutures_1min_varity_path_dict.get(variety, []) + [symbol_path]

allfutures_1min_varity_path_dict

E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\IC
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\IF
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\IH
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\IM
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\T
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\TF
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\TL
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\CCFX\TS
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\GFEX
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\GFEX\LC
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\GFEX\PS
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\GFEX\SI
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\XDCE
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\XDCE\A
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\XDCE\B
E:\BaiduNetdiskDownload\期货全部合约一分钟-10月19\期货全部合约一分钟\XDCE\BB
E:\BaiduNetdiskDownload\期货

{'IC': ['E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1505.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1506.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1507.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1508.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1509.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1510.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1511.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1512.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1601.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1602.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1603.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合约一分钟-10月19\\期货全部合约一分钟\\CCFX\\IC\\IC1604.csv',
  'E:\\BaiduNetdiskDownload\\期货全部合

In [5]:
all_variety_list = []
for exchange, variety_list in exchange_variety_map.items():
    all_variety_list += variety_list

print(all_variety_list)
for variety in all_variety_list:

    # 品种文件夹路径 NED_variety_path
    NED_variety_path = os.path.join(native_exchange_data_dir, variety)

    # 时间序列文件夹路径 NED_variety_timeseries_path
    NED_variety_timeseries_path = os.path.join(NED_variety_path , "timeseries_data")

    # 合约状态表路径 NED_dimension_symbol_state_path
    NED_dimension_symbol_state_path = os.path.join(NED_variety_timeseries_path, 'dimension_symbol_state.csv')

    # 分钟行情时间序列文件夹路径 NED_futures_minutely_timeseries_path
    NED_futures_minutely_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_minutely")
    if os.path.exists(NED_dimension_symbol_state_path):  print(True)
    else:  print(False)


['T', 'TF', 'IF', 'IM', 'IC', 'IH', 'TS', 'TL', 'TA', 'ZC', 'TC', 'WT', 'CJ', 'PL', 'SR', 'CF', 'CY', 'FG', 'JR', 'LR', 'MA', 'OI', 'PF', 'PM', 'RI', 'RM', 'RS', 'SA', 'SF', 'SM', 'UR', 'WH', 'AP', 'WS', 'SH', 'ME', 'RO', 'PK', 'ER', 'PR', 'PX', 'JD', 'CS', 'JM', 'C', 'M', 'V', 'EB', 'EG', 'P', 'Y', 'L', 'FB', 'A', 'B', 'BB', 'I', 'J', 'PP', 'RR', 'LH', 'PG', 'LG', 'L_F', 'PP_F', 'V_F', 'BZ', 'LC', 'SI', 'PS', 'PD', 'PT', 'NR', 'SC', 'LU', 'BC', 'SCTAS', 'EC', 'AO', 'NI', 'AL', 'HC', 'BU', 'RU', 'FU', 'CU', 'AG', 'ZN', 'AU', 'PB', 'SS', 'RB', 'SN', 'WR', 'SP', 'BR', 'OP', 'AD']
True
True
True
True
True
True
True
True
False
False
True
False
True
True
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
T

In [6]:
for variety in allfutures_1min_varity_path_dict.keys():

    # 品种文件夹路径 NED_variety_path
    NED_variety_path = os.path.join(native_exchange_data_dir, variety)

    # 时间序列文件夹路径 NED_variety_timeseries_path
    NED_variety_timeseries_path = os.path.join(NED_variety_path , "timeseries_data")

    # 合约状态表路径 NED_dimension_symbol_state_path
    NED_dimension_symbol_state_path = os.path.join(NED_variety_timeseries_path, 'dimension_symbol_state.csv')

    # 分钟行情时间序列文件夹路径 NED_futures_minutely_timeseries_path
    NED_futures_minutely_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_minutely_timeseries")

    print(variety, end = '\t')
    if os.path.exists(NED_variety_path):  print(True)
    else:
        print(False)
        continue



IC	True
IF	True
IH	True
IM	True
T	True
TF	True
TL	True
TS	True
LC	True
PS	True
SI	True
A	True
B	True
BB	True
BZ	True
C	True
CS	True
EB	True
EG	True
FB	True
I	True
J	True
JD	True
JM	True
L	True
LG	True
LH	True
M	True
P	True
PG	True
PP	True
RR	True
V	True
Y	True
BC	True
EC	True
LU	True
NR	True
SC	True
AD	True
AG	True
AL	True
AO	True
AU	True
BR	True
BU	True
CU	True
FU	True
HC	True
NI	True
OP	True
PB	True
RB	True
RU	True
SN	True
SP	True
SS	True
WR	True
ZN	True
AP	True
CF	True
CJ	True
CY	True
ER	True
FG	True
GN	True
JR	True
LR	True
MA	True
ME	True
OI	True
PF	True
PK	True
PL	True
PM	True
PR	True
PX	True
RI	True
RM	True
RO	True
RS	True
SA	True
SF	True
SH	True
SM	True
SR	True
TA	True
TC	True
UR	True
WH	True
WS	True
WT	True
ZC	True


In [20]:
import shutil

print("\n开始迁移合约文件...")

for variety, symbol_path_list in allfutures_1min_varity_path_dict.items():
    # 构造目标路径（保持与上文一致的命名规范）
    NED_variety_path = os.path.join(native_exchange_data_dir, variety)
    NED_variety_timeseries_path = os.path.join(NED_variety_path, "timeseries_data")
    NED_futures_minutely_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_minutely_timeseries")

    # 确保目标目录存在（递归创建）
    if not os.path.exists(NED_futures_minutely_timeseries_path):
        os.makedirs(NED_futures_minutely_timeseries_path, exist_ok = True)
        print(f"创建目录: {NED_futures_minutely_timeseries_path}")

    # 迁移该品种下的所有合约文件
    for symbol_path in symbol_path_list:
        symbol_filename = os.path.basename(symbol_path)
        dest_path = os.path.join(NED_futures_minutely_timeseries_path, symbol_filename)

        try:
            # 使用copy2保留元数据，如不需要保留源文件可改为shutil.move
            if not os.path.exists(dest_path):
                shutil.copy2(symbol_path, dest_path)
                print(f"[复制] {variety}/{symbol_filename}")
            else:
                print(f"[跳过] {variety}/{symbol_filename} 已存在")
        except Exception as e:
            print(f"[错误] {variety}/{symbol_filename}: {str(e)}")

print("迁移完成")


开始迁移合约文件...
创建目录: E:\Latitude_Analytics\03_Futures_Database\Native_Exchange_Data\IC\timeseries_data\futures_minutely
[复制] IC/IC1505.csv
[复制] IC/IC1506.csv
[复制] IC/IC1507.csv
[复制] IC/IC1508.csv
[复制] IC/IC1509.csv
[复制] IC/IC1510.csv
[复制] IC/IC1511.csv
[复制] IC/IC1512.csv
[复制] IC/IC1601.csv
[复制] IC/IC1602.csv
[复制] IC/IC1603.csv
[复制] IC/IC1604.csv
[复制] IC/IC1605.csv
[复制] IC/IC1606.csv
[复制] IC/IC1607.csv
[复制] IC/IC1608.csv
[复制] IC/IC1609.csv
[复制] IC/IC1610.csv
[复制] IC/IC1611.csv
[复制] IC/IC1612.csv
[复制] IC/IC1701.csv
[复制] IC/IC1702.csv
[复制] IC/IC1703.csv
[复制] IC/IC1704.csv
[复制] IC/IC1705.csv
[复制] IC/IC1706.csv
[复制] IC/IC1707.csv
[复制] IC/IC1708.csv
[复制] IC/IC1709.csv
[复制] IC/IC1710.csv
[复制] IC/IC1711.csv
[复制] IC/IC1712.csv
[复制] IC/IC1801.csv
[复制] IC/IC1802.csv
[复制] IC/IC1803.csv
[复制] IC/IC1804.csv
[复制] IC/IC1805.csv
[复制] IC/IC1806.csv
[复制] IC/IC1807.csv
[复制] IC/IC1808.csv
[复制] IC/IC1809.csv
[复制] IC/IC1810.csv
[复制] IC/IC1811.csv
[复制] IC/IC1812.csv
[复制] IC/IC1901.csv
[复制] IC/IC1902.csv
[复制] IC/

In [ ]:
r"E:\Latitude_Analytics\03_Futures_Database\Native_Exchange_Data\RB\timeseries_data\futures_minutely"
r"E:\Latitude_Analytics\03_Futures_Database\Native_Exchange_Data\RB\timeseries_data\futures_minutely_timeseries"

In [7]:
import os
import shutil

print("开始路径修正：删除错误B路径，重命名A为B...")

# 遍历所有品种（基于之前代码中的字典）
for variety in allfutures_1min_varity_path_dict.keys():

    # 构造timeseries_data路径
    NED_variety_path = os.path.join(native_exchange_data_dir, variety)
    NED_variety_timeseries_path = os.path.join(NED_variety_path, "timeseries_data")

    # 如果上层目录不存在则跳过
    if not os.path.exists(NED_variety_timeseries_path):
        continue

    # A路径 (当前存在的正确数据文件夹，但命名不符)
    path_A = os.path.join(NED_variety_timeseries_path, "futures_minutely")
    # B路径 (错误创建的文件夹名，可能为空或需要清理)
    path_B = os.path.join(NED_variety_timeseries_path, "futures_minutely_timeseries")

    # 步骤1: 删除错误的B路径（如果存在）
    if os.path.exists(path_B):
        # ignore_errors=True 确保权限问题不会中断流程
        shutil.rmtree(path_B, ignore_errors=True)
        print(f"[清理] 已删除错误路径: {variety}/futures_minutely_timeseries")

    # 步骤2: 将A重命名为B（轻量级操作，仅修改目录项，不复制文件数据）
    if os.path.exists(path_A):
        os.rename(path_A, path_B)
        print(f"[修正] 已重命名: {variety}/futures_minutely -> futures_minutely_timeseries")
    else:
        print(f"[跳过] {variety}: A路径不存在")

print("路径修正完成")

开始路径修正：删除错误B路径，重命名A为B...
[修正] 已重命名: IC/futures_minutely -> futures_minutely_timeseries
[修正] 已重命名: IF/futures_minutely -> futures_minutely_timeseries
[修正] 已重命名: IH/futures_minutely -> futures_minutely_timeseries
[修正] 已重命名: IM/futures_minutely -> futures_minutely_timeseries
[清理] 已删除错误路径: T/futures_minutely_timeseries
[修正] 已重命名: T/futures_minutely -> futures_minutely_timeseries
[修正] 已重命名: TF/futures_minutely -> futures_minutely_timeseries
[修正] 已重命名: TL/futures_minutely -> futures_minutely_timeseries
[修正] 已重命名: TS/futures_minutely -> futures_minutely_timeseries
[清理] 已删除错误路径: LC/futures_minutely_timeseries
[修正] 已重命名: LC/futures_minutely -> futures_minutely_timeseries
[清理] 已删除错误路径: PS/futures_minutely_timeseries
[修正] 已重命名: PS/futures_minutely -> futures_minutely_timeseries
[清理] 已删除错误路径: SI/futures_minutely_timeseries
[修正] 已重命名: SI/futures_minutely -> futures_minutely_timeseries
[清理] 已删除错误路径: A/futures_minutely_timeseries
[修正] 已重命名: A/futures_minutely -> futures_minutely_timeseries
[清理] 已删除错误